# 05 — A simple baseline (Phase 5)

## Why a baseline?

Before anything clever, we build the **simplest system that works end to end** and score it.
That number becomes the bar every later idea must beat. Without it we can't tell whether a new idea helps.

## The plan

```
S1 entity ──► basic cleaning ──► blocking: share a RARE word ──► top 50 candidates
                                                                     │
                     name similarity + address similarity  ◄─────────┘
                                   │
                          score ≥ threshold ?  ──► predicted matches
```

Every step is deliberately crude. Phases 6–10 improve them one at a time.

## Two sets of S1 entities

| Set | Size | Used for |
|---|---|---|
| **tuning set** | 20,000 random *train* entities | choosing the threshold |
| **small validation** | 20,000 validation entities | measuring the final score |

We choose the threshold on *train* entities so the validation score stays an honest measurement (no leakage).

In [1]:
import gc
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.evaluation import candidate_recall, macro_f05, score_per_entity, score_report
from src.splits import make_validation_split

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 220)
TRAIN_DIR = PROJECT_ROOT / "dataset" / "train"
start = time.time()

In [2]:
gt = pd.read_csv(TRAIN_DIR / "train_ground_truth.tsv", sep="\t", keep_default_na=False)
split = make_validation_split(gt)

tuning_ids = split.loc[split["split"] == "train", "source1_entity_id"].sample(20_000, random_state=7)
validation_ids = split.loc[split["small_validation"], "source1_entity_id"]

truth = {
    "tuning": gt[gt["source1_entity_id"].isin(tuning_ids)].reset_index(drop=True),
    "validation": gt[gt["source1_entity_id"].isin(validation_ids)].reset_index(drop=True),
}
query_set = pd.concat([
    pd.DataFrame({"entity_id": tuning_ids, "query_set": "tuning"}),
    pd.DataFrame({"entity_id": validation_ids, "query_set": "validation"}),
])
query_set["query_set"].value_counts()

query_set
tuning        20000
validation    20000
Name: count, dtype: int64

---
## 1. Basic cleaning

Minimal and safe, based on Phase 2:

1. **lowercase** — `DAVIS FAMILY OFFICE` = `Davis Family Office`
2. **symbols → space** — `[Partners]`, `L.L.C.`, `***` (letters, digits and Indian vowel marks are kept, see the regex trap in notebook 02)
3. **squeeze spaces** — `Suresh  Broadcasters` → `suresh broadcasters`

No abbreviations, no accent removal, no suffix handling yet: those are Phase 6 experiments.

In [3]:
def basic_clean(text):
    return (
        text.str.lower()
        .str.replace(r"[^\p{L}\p{N}\p{M}\s]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

examples = pd.Series(["*** Borough Of South Corporation", "Terri Taylor Peak LLC [Partners]",
                      "5 MARY ELLA COURT, null, SMRYNA, DE", "राम मार्केटिंग प्राइवेट लिमिटेड", "Davis Family (Office)"])
pd.DataFrame({"before": examples, "after": basic_clean(examples)})

,before,after
0,*** Borough Of South Corporation,borough of south corporation
1,Terri Taylor Peak LLC [Partners],terri taylor peak llc partners
2,"5 MARY ELLA COURT, null, SMRYNA, DE",5 mary ella court null smryna de
3,राम मार्केटिंग प्राइवेट लिमिटेड,राम मार्केटिंग प्राइवेट लिमिटेड
4,Davis Family (Office),davis family office


Load the 40,000 query entities from Source 1 and turn each into a **set of words** (name words + address words).

### Memory lesson: numbers are cheaper than text

Our first version of this notebook crashed with `ArrowMemoryError: malloc of size 6746019072 failed`
(it asked for 6.7 GB at once). It joined tens of millions of rows on **text** columns.
The fix used by every large-scale system:

1. give every distinct `country|word` key an **integer code** (`pd.Index.get_indexer` looks the codes up),
   and give every record a **row number**; joins on 4-byte integers are far lighter than joins on text;
2. process the query entities in **batches**, keeping only the top candidates of each batch;
3. do **not** keep the text of all 10.3M pool records — after blocking, re-read the files and keep only the candidates.

In [4]:
s1 = pd.read_csv(TRAIN_DIR / "train_source1.tsv", sep="\t", keep_default_na=False)
s1 = s1.merge(query_set, on="entity_id").reset_index(drop=True)
s1["name_clean"] = basic_clean(s1["business_name"])
s1["addr_clean"] = basic_clean(s1["business_address"])
gc.collect()

s1_words = (
    s1[["country"]]
    .assign(word=(s1["name_clean"] + " " + s1["addr_clean"]).str.split())
    .explode("word")
    .dropna()
)
s1_words["key"] = s1_words["country"] + "|" + s1_words["word"]   # the country is part of the key
s1_words["s1_row"] = s1_words.index                               # row number in `s1`
s1_words = s1_words[["s1_row", "key"]].drop_duplicates()

vocabulary = pd.Index(s1_words["key"].unique())                   # key -> integer code = position in this Index
s1_words["key_code"] = vocabulary.get_indexer(s1_words["key"]).astype("int32")
print(f"{len(s1):,} query entities | {len(s1_words):,} (entity, word) rows | {len(vocabulary):,} distinct keys")
s1_words.head(8)

40,000 query entities | 466,652 (entity, word) rows | 59,030 distinct keys


,s1_row,key,key_code
0,0,US|nexus,0
0,0,US|anchor,1
0,0,US|rain,2
0,0,US|1111,3
0,0,US|church,4
0,0,US|street,5
0,0,US|unit,6
0,0,US|2007,7


---
## 2. Read the S2/S3 pool in chunks

We read each file **1 million rows at a time** (`chunksize=`). From each chunk we keep only
(pool row number, key code) for words that **also appear in our query entities** — other words can never link anything —
plus the list of entity IDs so a row number can be turned back into an ID.

**Another memory trick:** a key that appears more than `MAX_DF` times inside a *single* chunk is certainly too common
for blocking (see section 3), so we drop it right away and remember that we dropped it.

In [5]:
MAX_DF = 1000        # a word in more than 1000 pool records is "too common" for blocking

pool_ids, pool_words = [], []
too_common = set()
offset = 0

for source_number in [2, 3]:
    path = TRAIN_DIR / f"train_source{source_number}.tsv"
    for chunk in pd.read_csv(path, sep="\t", keep_default_na=False, chunksize=1_000_000):
        chunk = chunk.reset_index(drop=True)
        pool_ids.append(chunk["entity_id"])

        words = (
            chunk[["country"]]
            .assign(word=(basic_clean(chunk["business_name"]) + " " + basic_clean(chunk["business_address"])).str.split())
            .explode("word")
            .dropna()
        )
        codes = vocabulary.get_indexer(words["country"] + "|" + words["word"])   # -1 = not a query word
        words = pd.DataFrame({
            "pool_row": (words.index.to_numpy() + offset).astype("int32"),
            "key_code": codes.astype("int32"),
        })
        words = words[words["key_code"] >= 0].drop_duplicates()

        counts = words["key_code"].value_counts()
        too_common |= set(counts[counts > MAX_DF].index)
        pool_words.append(words[~words["key_code"].isin(too_common)])
        offset += len(chunk)
    print(f"source {source_number} done  ({time.time() - start:.0f}s)")

pool_ids = pd.concat(pool_ids, ignore_index=True)       # pool_ids[row number] -> entity_id
pool_words = pd.concat(pool_words, ignore_index=True)
pool_words = pool_words[~pool_words["key_code"].isin(too_common)]
gc.collect()
print(f"pool records: {len(pool_ids):,} | (record, word) rows kept: {len(pool_words):,} "
      f"| memory of that table: {pool_words.memory_usage().sum() / 1e6:.0f} MB")

source 2 done  (112s)


source 3 done  (203s)


pool records: 10,320,219 | (record, word) rows kept: 28,495,307 | memory of that table: 456 MB


---
## 3. Common words vs rare words (document frequency)

**Document frequency (df)** of a word = in how many pool records it appears.

- `limited` or `llc` appear in hundreds of thousands of records. Sharing them says almost nothing.
- `colombier` or `mylapore` appear in a handful. Sharing them is strong evidence.

**IDF** ("inverse document frequency") turns this into a weight: `idf = log(N / df)`.
Rare word → big weight, common word → small weight.

| word | df | idf = log(10.3M / df) |
|---|---|---|
| appears in 1 record | 1 | 16.1 |
| appears in 1,000 records | 1,000 | 9.2 |
| appears in 500,000 records | 500,000 | 3.0 |

In [6]:
N_POOL = len(pool_ids)
df = pool_words["key_code"].value_counts().rename("df")
df = df[df <= MAX_DF]
idf = np.log(N_POOL / df)                     # indexed by key_code

pool_words = pool_words[pool_words["key_code"].isin(df.index)]
s1_rare = s1_words[s1_words["key_code"].isin(df.index)]

print("df of the words kept for blocking:")
print(df.describe().round(1).to_string())
print(f"\nkeys dropped as too common: {len(too_common):,}, e.g.", list(vocabulary[sorted(too_common)[:12]]))
print("examples of rare keys:", list(vocabulary[df.sample(8, random_state=0).index]))

df of the words kept for blocking:
count    50165.0
mean       185.1
std        221.4
min          1.0
25%         25.0
50%        102.0
75%        251.0
max       1000.0

keys dropped as too common: 1,348, e.g. ['US|anchor', 'US|church', 'US|street', 'US|unit', 'US|nashville', 'US|tn', 'US|corner', 'US|26', 'US|stone', 'US|3', 'US|ma', 'India|care']
examples of rare keys: ['India|976', 'India|shoppe', 'US|leatherman', 'India|anukul', 'US|rounseville', 'US|whigham', 'India|adda', 'India|sno18']


---
## 4. Blocking: candidates = pool records that share a rare word

For each batch of 2,000 query entities:

1. join their word codes with the pool word codes (`merge` on `key_code`, which already contains the country);
2. for every (entity, record) pair, add up the **idf of the shared words** — more / rarer shared words = better candidate;
3. keep the **50 best** records per entity.

In [7]:
TOP_K = 50
BATCH_SIZE = 2_000

batches = []
for batch_start in range(0, len(s1), BATCH_SIZE):
    batch = s1_rare[(s1_rare["s1_row"] >= batch_start) & (s1_rare["s1_row"] < batch_start + BATCH_SIZE)]
    shared = batch.merge(pool_words, on="key_code")
    shared["idf"] = idf.reindex(shared["key_code"]).to_numpy()
    top = (
        shared.groupby(["s1_row", "pool_row"])
        .agg(shared_words=("idf", "size"), idf_sum=("idf", "sum"))
        .reset_index()
        .sort_values("idf_sum", ascending=False)
        .groupby("s1_row")
        .head(TOP_K)
    )
    batches.append(top)

candidates = pd.concat(batches, ignore_index=True)
candidates["source1_entity_id"] = s1["entity_id"].to_numpy()[candidates["s1_row"]]
candidates["candidate_id"] = pool_ids.to_numpy()[candidates["pool_row"]]
del batches, shared
gc.collect()
print(f"candidate pairs: {len(candidates):,}  ({time.time() - start:.0f}s)")
candidates.head()

candidate pairs: 1,669,877  (225s)


,s1_row,pool_row,shared_words,idf_sum,source1_entity_id,candidate_id
0,1446,146145,9,107.833438,S1-119887597,S2-496843998
1,1446,6255610,8,96.589097,S1-119887597,S3-494743233
2,1446,9001925,8,95.761360,S1-119887597,S3-451750777
3,1793,2785683,7,93.265164,S1-557145898,S2-822206889
4,1919,1572658,8,92.182485,S1-893222242,S2-640145486


### How good is this blocking? → **candidate recall**

Candidate recall = share of true pairs that made it into the candidate list.
It is a **ceiling**: a true match that is not a candidate can never be predicted.

In [8]:
def to_submission(pairs, id_column, out_column):
    return (pairs.groupby("source1_entity_id")[id_column].agg(",".join)
            .rename(out_column).reset_index())

candidate_lists = to_submission(candidates, "candidate_id", "candidate_entity_ids")
per_entity = candidates.groupby("source1_entity_id").size()

for name, t in truth.items():
    print(f"{name:>10}: candidate recall = {candidate_recall(candidate_lists, t):.3f}")
print("\ncandidates per entity:")
print(per_entity.describe().round(1).to_string())
print("entities with no candidate at all:", len(s1) - len(per_entity))

    tuning: candidate recall = 0.580


validation: candidate recall = 0.588

candidates per entity:
count    35078.0
mean        47.6
std          9.0
min          1.0
25%         50.0
50%         50.0
75%         50.0
max         50.0
entities with no candidate at all: 4922


Only about **60%** of true matches survive this blocking. That is the **biggest weakness** of the baseline,
and the main target of Phase 7. (Recall is similar for both sets, as it should be.)

---
## 5. Similarity: Jaccard on words

### Concept
Jaccard similarity of two word sets = **words in common ÷ all distinct words**.

### Why do we need it?
It is a number between 0 and 1 saying how much two texts overlap, ignoring **word order** (addresses get reordered)
and **case/symbols** (already cleaned).

### Simple example
```
A = {abc, restaurant, vijayawada}
B = {abc, rest, vijayawada}
common = {abc, vijayawada} → 2       all = {abc, restaurant, rest, vijayawada} → 4
Jaccard = 2 / 4 = 0.5
```

### Limitations
A typo makes a word completely different (`restaurant` vs `restarant` count as 0 overlap).
Character-level similarity (Phase 8) handles that.

### How we combine
`score = average of name Jaccard and address Jaccard` (if the candidate has **no address**, we use the name Jaccard alone).

In [9]:
def jaccard(a, b):
    a, b = set(a), set(b)
    if not a or not b:
        return np.nan
    return len(a & b) / len(a | b)

print(jaccard("abc restaurant vijayawada".split(), "abc rest vijayawada".split()))

0.5


In [10]:
# second pass over the pool: keep and clean only the candidate records
wanted = set(candidates["candidate_id"])
candidate_text = []
for source_number in [2, 3]:
    path = TRAIN_DIR / f"train_source{source_number}.tsv"
    for chunk in pd.read_csv(path, sep="\t", keep_default_na=False, chunksize=1_000_000,
                             usecols=["entity_id", "business_name", "business_address"]):
        chunk = chunk[chunk["entity_id"].isin(wanted)]
        candidate_text.append(pd.DataFrame({
            "entity_id": chunk["entity_id"],
            "name_clean": basic_clean(chunk["business_name"]),
            "addr_clean": basic_clean(chunk["business_address"]),
        }))
candidate_text = pd.concat(candidate_text, ignore_index=True)

pairs = (
    candidates[["source1_entity_id", "candidate_id", "shared_words", "idf_sum"]]
    .merge(s1[["entity_id", "query_set", "name_clean", "addr_clean"]], left_on="source1_entity_id", right_on="entity_id")
    .drop(columns="entity_id")
    .merge(candidate_text, left_on="candidate_id", right_on="entity_id", suffixes=("_s1", "_c"))
    .drop(columns="entity_id")
)
pairs["name_jaccard"] = [jaccard(a.split(), b.split()) for a, b in zip(pairs["name_clean_s1"], pairs["name_clean_c"])]
pairs["addr_jaccard"] = [jaccard(a.split(), b.split()) for a, b in zip(pairs["addr_clean_s1"], pairs["addr_clean_c"])]
pairs["score"] = pairs[["name_jaccard", "addr_jaccard"]].mean(axis=1).fillna(0)

# label each candidate pair with the truth (only for analysis, never as a feature)
true_pair_keys = set()
for t in truth.values():
    for s1_id, ids in zip(t["source1_entity_id"], t["matched_entity_ids"]):
        true_pair_keys.update((s1_id, m) for m in ids.split(",") if m)
pairs["is_true"] = [(a, b) in true_pair_keys for a, b in zip(pairs["source1_entity_id"], pairs["candidate_id"])]

del candidate_text, pool_words
gc.collect()
print(f"scored {len(pairs):,} pairs ({time.time() - start:.0f}s)")
pairs[["source1_entity_id", "candidate_id", "name_clean_s1", "name_clean_c", "name_jaccard", "addr_jaccard", "score", "is_true"]].head(8)

scored 1,669,877 pairs (462s)


,source1_entity_id,candidate_id,name_clean_s1,name_clean_c,name_jaccard,addr_jaccard,score,is_true
0,S1-119887597,S2-496843998,bz fabtex corporation,corporation bz fabtex,1.000000,0.777778,0.888889,True
1,S1-119887597,S3-494743233,bz fabtex corporation,bz services corp,0.200000,0.764706,0.482353,False
2,S1-119887597,S3-451750777,bz fabtex corporation,bz fabtex corporation,1.000000,0.666667,0.833333,True
3,S1-557145898,S2-822206889,united management private limited,यूनाइटेड मैनेजमेंट प्राइवेट लिमिटेड,0.000000,1.000000,0.500000,True
4,S1-893222242,S2-640145486,saru exports private limited,saru exports private ltd,0.600000,0.941176,0.770588,True
5,S1-893222242,S2-14455184,saru exports private limited,saru exports private,0.750000,0.882353,0.816176,True
6,S1-893222242,S3-750214688,saru exports private limited,mr saru ventures private limited sérvice,0.428571,0.777778,0.603175,False
7,S1-116805383,S3-381156005,ramaiah indiaprivate india pvt ltd,pvt ramaiah indiaprivate india ltd,1.000000,0.842105,0.921053,True


### Do true pairs score higher than false ones?

`pd.cut` puts scores into buckets; we count true and false candidate pairs per bucket (tuning set only).

In [11]:
tuning_pairs = pairs[pairs["query_set"] == "tuning"]
buckets = pd.cut(tuning_pairs["score"], bins=np.arange(0, 1.01, 0.1), include_lowest=True)
table = pd.crosstab(buckets, tuning_pairs["is_true"]).rename(columns={False: "false pairs", True: "true pairs"})
table["share true %"] = (table["true pairs"] / table.sum(axis=1) * 100).round(1)
table

is_true,false pairs,true pairs,share true %
score,,,
"(-0.001, 0.1]",518855,29,0.0
"(0.1, 0.2]",190373,370,0.2
"(0.2, 0.3]",52274,931,1.7
"(0.3, 0.4]",17079,2530,12.9
"(0.4, 0.5]",8003,5303,39.9
"(0.5, 0.6]",3668,5285,59.0
"(0.6, 0.7]",2373,8321,77.8
"(0.7, 0.8]",1106,8691,88.7
"(0.8, 0.9]",150,5657,97.4


True pairs concentrate at high scores and false pairs at low scores, but they overlap in the middle.
The threshold decides where we cut.

---
## 6. Choose the threshold on the tuning set

For each threshold `t`: predict every candidate with `score ≥ t`, compute macro F0.5 on the **tuning** entities.

In [12]:
def predictions_at(pair_table, threshold):
    chosen = pair_table[pair_table["score"] >= threshold]
    return to_submission(chosen, "candidate_id", "matched_entity_ids")

rows = []
for threshold in np.arange(0.20, 0.96, 0.05):
    report = score_report(predictions_at(tuning_pairs, threshold), truth["tuning"])
    rows.append({"threshold": round(threshold, 2), **report.to_dict()})
threshold_table = pd.DataFrame(rows).set_index("threshold").round(4)
threshold_table

,macro_F0.5,F0.5 on entities with matches,F0.5 on singletons,pair precision,pair recall,predicted pairs,true pairs
threshold,,,,,,,
0.20,0.3630,0.3674,0.2884,0.2996,0.5745,132471.0,69079.0
0.25,0.4222,0.4266,0.3468,0.4002,0.5698,98373.0,69079.0
0.30,0.4715,0.4765,0.3872,0.5168,0.5616,75059.0,69079.0
0.35,0.5071,0.5120,0.4232,0.6276,0.5488,60407.0,69079.0
0.40,0.5194,0.5231,0.4564,0.6869,0.5273,53033.0,69079.0
0.45,0.5234,0.5249,0.4969,0.7359,0.4982,46767.0,69079.0
0.50,0.5227,0.5212,0.5481,0.7568,0.4728,43159.0,69079.0
0.55,0.5217,0.5135,0.6604,0.8340,0.4201,34791.0,69079.0
0.60,0.5077,0.4934,0.7511,0.8656,0.3762,30021.0,69079.0


In [13]:
best_threshold = threshold_table["macro_F0.5"].idxmax()
print("best threshold on the tuning set:", best_threshold, "-> tuning F0.5 =", threshold_table.loc[best_threshold, "macro_F0.5"])

best threshold on the tuning set: 0.45 -> tuning F0.5 = 0.5234


Note how precision rises and recall falls as the threshold goes up; F0.5 peaks where precision is high
but recall hasn't collapsed yet.

---
## 7. The baseline score on the small validation set

We apply the threshold chosen above — **without** touching it — to the validation entities.

In [14]:
validation_pairs = pairs[pairs["query_set"] == "validation"]
validation_predictions = predictions_at(validation_pairs, best_threshold)
baseline_report = score_report(validation_predictions, truth["validation"])
baseline_report.round(4)

macro_F0.5                           0.5278
F0.5 on entities with matches        0.5290
F0.5 on singletons                   0.5078
pair precision                       0.7381
pair recall                          0.5059
predicted pairs                  47284.0000
true pairs                       68984.0000
dtype: float64

In [15]:
per_entity_scores = score_per_entity(validation_predictions, truth["validation"]).merge(
    s1[["entity_id", "country"]], left_on="source1_entity_id", right_on="entity_id"
)
per_entity_scores.groupby("country")["f05"].agg(["mean", "size"]).round(4)

,mean,size
country,,
India,0.5420,8039
US,0.5182,11961


---
## 8. A first look at the mistakes

### Wrong matches (false positives) — the expensive kind

In [16]:
cols = ["name_clean_s1", "name_clean_c", "addr_clean_s1", "addr_clean_c", "score"]
fp = validation_pairs[(validation_pairs["score"] >= best_threshold) & ~validation_pairs["is_true"]]
print(f"false positive pairs: {len(fp):,}")
fp[cols].sample(10, random_state=0)

false positive pairs: 12,385


,name_clean_s1,name_clean_c,addr_clean_s1,addr_clean_c,score
1085307,prairie anchor home care llc,prairie anchor home care ltd,barnard 508 stack pole road vt,barnard 517 stack pole rd vermont,0.500000
1339714,conroy legacy ithax llc,conroy legacy ithax inc,virginia beach city 421 103 lynnhaven road va,421 116a lynnhaven rd virginia beach virginia,0.500000
836652,graviss fragrance,graviss fragrance,m 5 shekhar central 4 5 manorama ganj palasia squ indore...,unit no 04 1st floor no 18 cunningham road bangalore nor...,0.500000
1502812,latur timbers limited,latur timbers llp,c o subhash rambhau dalvi house no 3450 latur maharashtra,no 22 c o subhash rambhau dalvi house no 3450 latur maha...,0.704545
433291,ahmedabad beads private limited,ink beads private limited,d 40 b g towers o s delhigate shahibaug ahmedabad gujarat,,0.600000
672126,validus films,validus films group,bangalore karnataka 15 1 dickenson road bangalore,no 015 14 dickenson road bangalore bangalore ಕರ್ನಾಟಕ,0.483333
1344598,ankita industries private limited,ariasynvio,14 narayan nagar lal singh judo colony tonk road jaipur ...,14 narayan nagar lal singh judo colony tonk road jaipur ...,0.500000
1320184,20 20 optical,20 20 optical,2408 fillmore street city of marshfield wi,519 2st ave dixon il,0.500000
1418451,properties evergreen india private limited,properties evergreen building private limited,12103 presige falocon city bangalore karnataka,12110 presige falocon city thurahalli road bangalore ka,0.533333
333163,lsd infosystems india limited,lsd infosystems india ventures ventures,opposite sena medal ksk petrol pump near sector 19 20 hs...,no 0062 opposite sena medal ksk petrol pump near sector ...,0.688889


### Missed matches that *were* candidates (score too low)

In [17]:
fn = validation_pairs[(validation_pairs["score"] < best_threshold) & validation_pairs["is_true"]]
print(f"true candidate pairs below the threshold: {len(fn):,}")
fn[cols].sample(10, random_state=0)

true candidate pairs below the threshold: 5,647


,name_clean_s1,name_clean_c,addr_clean_s1,addr_clean_c,score
669350,burke bulkers,zetapyra,18219 stavis bay road seabeck wa,18219 stavis bay rd seabeck washington,0.250000
93592,urban marketing private limited,अर्बन मार्केटिंग प्राइवेट लिमिटेड,house no 139 behind turtles school ashok vihar jagatpura...,house no 139 behind turtles school ashok vihar jagatpura...,0.416667
825519,bombay foundation llp,बॉम्बे फाउंडेशन एलएलपी,c o moradabad road opp ekta vihar sambhal moradabad samb...,moradabad sambhal उत्तर प्रदेश c o moradabad road opp ek...,0.333333
1594093,i s stiles premier inc,i s stiles,2321 18th street washington dc,washingtont ownship district of columbia 2321 18st st,0.341667
834087,alpha black global private limited,આલ્ફા બ્લેક ગ્લોબલ પ્રાઇવેટ લિમિટેડ,satta no hall main road viva city village surendranagar ...,satta no hall main road viva city village surendranagar ...,0.428571
586096,north hospitality private limited,northhospitality com,c o haribhajan dnynobaji kalse vinoba bhave nagar nagpur...,c o haribhajan dnynobaji kalse vinoba bhave nagar nagpur...,0.409091
1168775,creative estate private limited,क्रिएटिव एस्टेट प्राइवेट लिमिटेड,vill parsagargh ekma saran bihar saran,vill parsagargh ekma saran बिहार,0.333333
638735,piedmont allied plus llc,piedmontalliedplus com,725 hammonds lane fl 0 brooklyn md,725 hammonds lane brooklyn maryland,0.250000
1016466,safe daedalus,safedaedalus com,129 royal blackheath smithfield town va,va smithfield royal blackheath,0.333333
1243122,city services private limited,सिटी सर्विसेज प्राइवेट लिमिटेड,qtr no 229 g f 4 storey near police chowki new delhi wes...,hn 388 qtr no 229 new delhi west delhi delhi,0.200000


In [18]:
print(f"total run time: {time.time() - start:.0f}s")

total run time: 475s


---
# Summary — Phase 5

| | Value |
|---|---|
| Candidate recall (blocking ceiling) | 0.58 tuning / 0.59 validation |
| Entities with no candidate at all | 4,922 of 40,000 (all their words are "too common") |
| Threshold (chosen on the tuning set) | 0.45 |
| **Macro F0.5 on small validation** | **0.528** (tuning 0.523 — no sign of overfitting) |
| F0.5 on entities with matches / on singletons | 0.529 / 0.508 |
| Pair precision / pair recall | 0.738 / 0.506 |

What the mistakes tell us:
- **False merges** often differ only in the **legal suffix** (`llc` vs `ltd`, `llc` vs `inc`) or the **house number** (`508` vs `517 stack pole road`).
  Both are strong "different business" signals that Jaccard ignores.
- **Missed matches** are dominated by **Indian-script names** (name Jaccard = 0), **website names** (`piedmontalliedplus com`),
  **trade names** (`zetapyra`) and abbreviations (`street` vs `st`, `tennessee` vs `tn`).
- **Blocking recall (~59%) caps everything**: even a perfect scorer could not exceed it.

Next phases attack these one at a time and must beat **0.528**.
